# FinSafe — versioned recovery and output-budget diagnostic
Rebuilds lost runtime artifacts from pinned sources, re-measures validation baselines, recreates the five-step smoke checkpoint, then runs the capped-output sensitivity diagnostic. This does not start the 100-step run or score unreviewed safety responses. Requires the notebook's existing HF_TOKEN secret. All stages log timing/failures; generation and training checkpoint themselves. Final outputs are retained by the versioned Kaggle run. Reproducible dependency environments/caches are removed only after success to keep retained outputs compact.

In [ ]:
import os,sys,json,subprocess,time,datetime
from pathlib import Path
boot_start=time.monotonic()
print(f"[{datetime.datetime.now(datetime.timezone.utc).isoformat()}] START reproducible rebuild",flush=True)
project=Path('/kaggle/working/finsafe')
if not project.exists():subprocess.run(['git','clone','https://github.com/dixitkeshav/finsafe.git',str(project)],check=True)
sys.path.insert(0,str(project/'src'))
from finsafe.progress import Progress,atomic_json
progress=Progress(project/'runs/rebuild/logs','rebuild')
progress.log(f'DONE bootstrap in {time.monotonic()-boot_start:.1f}s')
def run_cells(name,indices):
    notebook=json.loads((project/'notebooks'/name).read_text())
    for index in indices:
        with progress.stage(f'{name}-cell-{index}'):
            exec(compile(''.join(notebook['cells'][index]['source']),f'{name}:{index}','exec'),globals())
with progress.stage('01-runtime'):
    print(sys.version,flush=True)
    subprocess.run(['nvidia-smi'],check=True)
run_cells('01_kaggle_tatqa.ipynb',[5,7,11])
with progress.stage('02-pinned-model-references'):
    pinned={'CohereLabs/tiny-aya-global':'af89d219b53ed9b13b8a4645f9c8028973510324','CohereLabs/tiny-aya-base':'d62ac17e241c871fc52e9a95d7b0e2615189d7d5'}
    atomic_json(project/'runs/environment/model-revisions.json',pinned)
    progress.log(json.dumps(pinned))
run_cells('02_kaggle_t4_baseline.ipynb',[1,2,4,5,6,8,9])
# Avoid changing the historical eight-example generation contract. Recreate it in
# a fresh directory and preserve the old result as historical evidence in docs.
with progress.stage('03-smoke-data'):
    from finsafe.data import read_jsonl,write_jsonl
    write_jsonl(project/'data/processed/tatqa-smoke.jsonl',read_jsonl(project/'data/processed/tatqa-tokenized/validation.jsonl')[:8])
run_cells('03_kaggle_training_smoke.ipynb',[1,2,4,6,7,9,11])
run_cells('04_kaggle_safety_baseline.ipynb',[1,2,3,5,6,7,8])
run_cells('04b_kaggle_output_budget.ipynb',[1,2,3,4,5,6])
with progress.stage('07-preserve-portable-outputs'):
    # These are reproducible runtime dependencies/caches, not trained adapters or
    # unique data. Exclude them from the versioned notebook's retained outputs.
    import shutil
    for disposable in [project/'.venv',project/'.venv-gpu',Path('/kaggle/working/.cache'),Path('/kaggle/working/.python')]:
        if disposable.exists():
            progress.log(f'Removing regenerated environment/cache from retained output: {disposable}')
            shutil.rmtree(disposable)
    atomic_json(project/'runs/rebuild/completed.json',{'status':'complete','source_revision':subprocess.check_output(['git','-C',str(project),'rev-parse','HEAD'],text=True).strip(),'safety_rates':None,'review_status':'UNREVIEWED'})
    progress.log('REBUILD COMPLETE. Versioned outputs contain data, response review packets, checkpoints and verified archives. Environments are reinstallable.')
